# Value Learning

参考王树森《深度强化学习》一书。

下面是一张图，描述了整个知识体系应该是怎样的。

![value learnig](assets/value_learning_arch.png)

## 1. 环境准备

这里来个最简单的 `CartPole-v1`.

In [1]:
import gymnasium as gym

env = gym.make("CartPole-v1")

print(env)
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

<TimeLimit<OrderEnforcing<PassiveEnvChecker<CartPoleEnv<CartPole-v1>>>>>
Observation space: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Action space: Discrete(2)


In [2]:
state, info = env.reset()

print("state =", state)
print("shape =", state.shape)
print("info =", info)

state = [ 0.01315221  0.00917715 -0.04624644  0.04766193]
shape = (4,)
info = {}


在这个系统中，有两个物理对象，一个小车，一个杆。状态向量有四维，分别对应小车位置、小车速度、杆的角度与杆的角速度。在这个状态编码下系统具有一阶马尔科夫性。而 action space 只有两个离散的值：`0` 和 `1`，分别对应向左与向右。下面体验一下，以尽快熟悉开发环境。

In [3]:
action = 0

next_state, reward, terminated, truncated, info = env.step(action)

print(f"next_state: {state}")
print(f"reward: {reward}")
print(f"terminated: {terminated}")
print(f"truncated: {truncated}")

env.close()

next_state: [ 0.01315221  0.00917715 -0.04624644  0.04766193]
reward: 1.0
terminated: False
truncated: False


注意，这里和旧书中的 `observation, reward, done, info = env.step(action)` 不一样。

接下来看看一个随机 agent. （懒得录屏了，后面自己把代码运行一下就看得到）

In [4]:
env = gym.make("CartPole-v1", render_mode="human")
state, info = env.reset()
total_reward = 0

for t in range(1000):
    action = env.action_space.sample()
    state, reward, terminated, truncated, info = env.step(action)
    total_reward += reward

    if terminated or truncated:
        break

print(total_reward)
env.close()

19.0


大致理解了整个流程，尝试一个简单的启发式 Agent:

In [5]:
env = gym.make("CartPole-v1", render_mode="human")
state, info = env.reset()
total_reward = 0

for t in range(1000):
    pole_angle = state[2]
    action = 0 if pole_angle < 0 else 1
    state, reward, terminated, truncated, info = env.step(action)
    total_reward += reward

    if terminated or truncated:
        break

print(total_reward)
env.close()

34.0


玩到这里，开始进入正题了

## 2. DQN

我们需要训练一个神经网络 $$ Q_\theta: \mathbb{R}^4 \to \mathbb{R}^2 $$，其根据当前 state 判断每个动作的价值。

下面给出完整代码。（懒得一个 Cell 一个 Cell 的写了，直接在 Neovim 中编辑好了 Load 进来，DRL 这部分应该都会这样，省时间；另外，还会使用大量 AI 生成，自己稍微修改的代码）

In [12]:
# %load dqn.py
import random
from collections import deque
import os

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim


# -------------------------
# Environment
# -------------------------

env = gym.make("CartPole-v1")

state_dim = env.observation_space.shape[0]  # 4
action_dim = env.action_space.n             # 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using device: {device}")

# -------------------------
# Q Network
# -------------------------

class DQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )

    def forward(self, x):
        return self.net(x)


q = DQN(state_dim, action_dim).to(device)
q_target = DQN(state_dim, action_dim).to(device)

q_target.load_state_dict(q.state_dict())

optimizer = optim.Adam(q.parameters(), lr=1e-3)


# -------------------------
# Replay Buffer
# -------------------------

buffer = deque(maxlen=10000)


# -------------------------
# Hyperparameters
# -------------------------

gamma = 0.99
batch_size = 64
epsilon = 1.0


# -------------------------
# Training
# -------------------------

def choose_action(state):
    # epsilon-greedy
    if random.random() < epsilon:
        return env.action_space.sample()
    
    state_tensor = torch.tensor(state, dtype=torch.float32, device=device)
    with torch.no_grad():
        return q(state_tensor).argmax().item()



for episode in range(300):

    state, _ = env.reset()
    total_reward = 0

    while True:
            
        action = choose_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)

        done = terminated or truncated

        # save transition
        buffer.append((state, action, reward, next_state, terminated))

        state = next_state
        total_reward += reward

        # train
        if len(buffer) >= batch_size:

            batch = random.sample(buffer, batch_size)

            states, actions, rewards, next_states, terminateds = zip(*batch)

            states = torch.tensor(np.array(states), dtype=torch.float32, device=device)
            actions = torch.tensor(actions, device=device).unsqueeze(1)
            rewards = torch.tensor(rewards, dtype=torch.float32, device=device)
            next_states = torch.tensor(np.array(next_states), dtype=torch.float32, device=device)
            terminateds = torch.tensor(terminateds, dtype=torch.float32, device=device)

            # Q(s, a)
            q_values = q(states).gather(1, actions).squeeze()

            # r + gamma max Q_target(s', a')
            with torch.no_grad():
                next_q_values = q_target(next_states).max(1).values
                targets = rewards + gamma * (1 - terminateds) * next_q_values
            loss = ((q_values - targets) ** 2).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if done:
            break

    # epsilon decay
    epsilon = max(0.05, epsilon * 0.98)

    # update target network
    if episode % 10 == 0:
        q_target.load_state_dict(q.state_dict())

    print(
        f"episode={episode}, "
        f"reward={total_reward}, "
        f"epsilon={epsilon:.2f}"
    )

os.makedirs("checkpoints", exist_ok=True)
torch.save(q.state_dict(), "checkpoints/dqn_cartpole.pt")
print("model saved to checkpoints/dqn_cartpole.pt")

env.close()


## 3. Deep SARSA

下面看看两个方法的对比：

- Q-Learning (离线)： $Q(S, A) \leftarrow Q(S, A) + \alpha [R + \gamma \max_{a'} Q(S', a') - Q(S, A)]$
- SARSA (在线)： $Q(S, A) \leftarrow Q(S, A) + \alpha [R + \gamma Q(S', A') - Q(S, A)]$

一大不同就是 Q-Learning 以估计最优为目标，而这个东西来自自己本身（这是一种自举）;而 SARSA 则是以当前执行情况为目标。前者可以使用经验回放而后者不行。

下面直接上代码：

In [14]:
# %load deep-sarsa.py
import random
import os

import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim


env = gym.make("CartPole-v1")

state_dim = env.observation_space.shape[0]   # 4
action_dim = env.action_space.n              # 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using device: {device}")

# -------------------------
# Q network
# -------------------------

class DQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
        )

    def forward(self, x):
        return self.net(x)

q = DQN(state_dim, action_dim).to(device)

optimizer = optim.Adam(q.parameters(), lr=1e-3)

gamma = 0.99
epsilon = 1.0

# -------------------------
# epsilon-greedy
# -------------------------

def choose_action(state):
    if random.random() < epsilon:
        return env.action_space.sample()

    state_tensor = torch.tensor(state, dtype=torch.float32, device=device)

    with torch.no_grad():
        return q(state_tensor).argmax().item()

# -------------------------
# SARSA training
# -------------------------

for episode in range(3000):
    
    state, _ = env.reset()
    action = choose_action(state)
    total_reward = 0

    while True:
    
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        if not terminated:
            next_action = choose_action(next_state)

        state_tensor = torch.tensor(state, dtype=torch.float32, device=device)
        q_values = q(state_tensor)[action]

        if terminated:
            target = torch.tensor(reward, dtype=torch.float32, device=device)
        else:
            next_state_tensor = torch.tensor(next_state, dtype=torch.float32, device=device)
            with torch.no_grad():
                target = reward + gamma * q(next_state_tensor)[next_action]

        # TD loss
        loss = (q_values - target) ** 2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        state = next_state

        if not terminated:
            action = next_action

        total_reward += reward

        if done:
            break

    epsilon = max(0.05, epsilon * 0.98)
    
    if (episode + 1) % 100 == 0:
        print(
            f"episode={episode}, "
            f"reward={total_reward}, "
            f"epsilon={epsilon:.2f}"
        )

os.makedirs("checkpoints", exist_ok=True)
torch.save(q.state_dict(), "checkpoints/sarsa_cartpole.pt")
print("model save to checkpoints/sarsa_cartpole.pt")

env.close()


下面是演示代码，支持演示两种不同的训练算法：

In [ ]:
# %load demo.py
# run dqn.py before you start dqn_demo.py

import argparse

import gymnasium as gym
import torch
import torch.nn as nn

class DQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
        )

    def forward(self, x):
        return self.net(x)

parser = argparse.ArgumentParser()
parser.add_argument("name", help="model name(dqn/sarsa)")
args = parser.parse_args()

model_path = None
if args.name == "dqn":
    model_path = "checkpoints/dqn_cartpole.pt"
elif args.name == "sarsa":
    model_path = "checkpoints/sarsa_cartpole.pt"
else:
    raise RuntimeError("model name doesn't exist!")

assert model_path is not None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using device: {device}")

env = gym.make("CartPole-v1", render_mode="human")

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

q = DQN(state_dim, action_dim).to(device)
q.load_state_dict(torch.load(model_path, map_location=device))
q.eval()

for episode in range(5):
    state, _ = env.reset()
    total_reward = 0

    while True:
        state_tensor = torch.tensor(state, dtype=torch.float32, device=device)

        with torch.no_grad():
            q_values = q(state_tensor)
            action = q_values.argmax().item()

        state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        if terminated or truncated:
            break

    print(
        f"episode={episode}, "
        f"reward={total_reward}"
    )

env.close()


这部分的代码演示就到这里。